# Supplementary Implementation: SGH-GraphBNN

This notebook provides a reproducible implementation for **SGH-GraphBNN: Stochastic Gradient Hamiltonian Graph Bayesian Neural Network for uncertainty-aware IoT intrusion detection**.

The workflow covers dataset loading, preprocessing, graph construction, deterministic GraphSAGE pretraining, cached edge-embedding extraction, low-rank SGHMC posterior sampling, posterior prediction, calibration, uncertainty estimation, OoD detection, five-seed robustness analysis, runtime and memory tracking, statistical comparison, and supplementary figure/table generation.

The notebook is distributed as supporting material without executed outputs. Set the dataset paths in the configuration cell before running the workflow.

In [ ]:
# ============================================================
# CELL 1: Environment Setup and Imports
# ============================================================
# Supplementary implementation for:
# SGH-GraphBNN: Stochastic Gradient Hamiltonian Graph Bayesian Neural Network
# for uncertainty-aware IoT intrusion detection.

import os
import gc
import json
import math
import time
import random
import warnings
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Any

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    log_loss,
    brier_score_loss,
    confusion_matrix,
    roc_curve,
    precision_recall_curve,
)
from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import PCA
from scipy.stats import wilcoxon

import matplotlib.pyplot as plt

try:
    import seaborn as sns
except Exception:
    sns = None

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import TensorDataset, DataLoader
except Exception as exc:
    raise ImportError("PyTorch is required to run SGH-GraphBNN.") from exc

try:
    from torch_geometric.data import Data
    from torch_geometric.nn import SAGEConv
    PYG_AVAILABLE = True
except Exception:
    PYG_AVAILABLE = False
    Data = None
    SAGEConv = None

warnings.filterwarnings("ignore")
plt.rcParams["figure.dpi"] = 130
plt.rcParams["savefig.dpi"] = 300


In [ ]:
# ============================================================
# CELL 2: Global Configuration and Reproducibility Controls
# ============================================================

FRAMEWORK_NAME = "SGH-GraphBNN"

@dataclass
class DatasetSpec:
    name: str
    file_path: str
    graph_mode: str
    label_column: str = "Label"
    source_ip_column: Optional[str] = None
    destination_ip_column: Optional[str] = None
    timestamp_column: Optional[str] = None
    max_rows: Optional[int] = None
    knn_neighbors: int = 8

@dataclass
class ExperimentConfig:
    output_dir: str = "outputs_sgh_graphbnn"
    seed: int = 42
    five_seed_values: Tuple[int, ...] = (42, 123, 2024, 3407, 777)
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    train_size: float = 0.80
    val_size: float = 0.08
    test_size: float = 0.12
    top_k_numeric_features: int = 32
    add_edge_l2_norm: bool = True
    hidden_dim: int = 128
    sage_layers: int = 2
    gnn_dropout: float = 0.10
    classifier_dropout: float = 0.25
    batch_size: int = 4096
    sghmc_batch_size: int = 8192
    max_epochs: int = 100
    early_stop_patience: int = 5
    learning_rate: float = 2e-3
    weight_decay: float = 1e-5
    label_smoothing: float = 0.01
    temperature_max_iter: int = 150
    mc_dropout_passes: int = 10
    sghmc_subset_size: int = 60000
    subset_sizes: Tuple[int, ...] = (15000, 20000, 80000)
    sghmc_chains: int = 2
    sghmc_iterations: int = 500
    sghmc_burn_in: int = 300
    sghmc_retained_per_chain: int = 200
    sghmc_step_size: float = 0.0184
    sghmc_friction: float = 0.051
    sghmc_noise_scale: float = 0.0100
    sghmc_prior_variance: float = 1.0
    low_rank_dim: int = 16
    calibration_bins: int = 10
    ood_quantile: float = 0.95
    max_graph_edges_for_similarity: int = 2_000_000
    save_figures: bool = True
    save_tables: bool = True

cfg = ExperimentConfig()
output_dir = Path(cfg.output_dir)
fig_dir = output_dir / "figures"
table_dir = output_dir / "tables"
model_dir = output_dir / "models"
for d in [output_dir, fig_dir, table_dir, model_dir]:
    d.mkdir(parents=True, exist_ok=True)

def set_global_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = False
        torch.backends.cudnn.benchmark = True

set_global_seed(cfg.seed)
DEVICE = torch.device(cfg.device)
print_device = str(DEVICE)


In [ ]:
# ============================================================
# CELL 3: Dataset Specifications
# ============================================================
# Update these paths to the local dataset locations before running.

DATA_ROOT = Path("./data")

nf_spec = DatasetSpec(
    name="NF-ToN-IoT-v3",
    file_path=str(DATA_ROOT / "NF-ToN-IoT-v3.csv"),
    graph_mode="directed_communication",
    label_column="Label",
    source_ip_column="IPV4_SRC_ADDR",
    destination_ip_column="IPV4_DST_ADDR",
    timestamp_column=None,
    max_rows=None,
)

cic_spec = DatasetSpec(
    name="CICIoT2023",
    file_path=str(DATA_ROOT / "CICIoT2023.csv"),
    graph_mode="flow_similarity",
    label_column="Label",
    source_ip_column=None,
    destination_ip_column=None,
    timestamp_column="Timestamp",
    max_rows=None,
    knn_neighbors=8,
)

DATASET_SPECS = [nf_spec, cic_spec]


In [ ]:
# ============================================================
# CELL 4: Dataset Profiling Utilities
# ============================================================
# These helpers compute dataset profiles directly from the files used in the run.
# They do not provide fixed benchmark outcomes.

def profile_dataset_csv(spec: DatasetSpec, chunksize: int = 1_000_000) -> pd.DataFrame:
    """Compute basic dataset profile information from the configured CSV file.

    The function streams the file in chunks so it can be used with large flow datasets.
    It reports row count, label counts, numeric feature count, and detected columns.
    """
    path = Path(spec.file_path)
    if not path.exists():
        raise FileNotFoundError(f"Dataset file not found: {path}")

    total_rows = 0
    label_counts = {}
    numeric_columns = set()
    all_columns = None

    for chunk in pd.read_csv(path, chunksize=chunksize, low_memory=False):
        if all_columns is None:
            all_columns = list(chunk.columns)
        total_rows += len(chunk)
        if spec.label_column in chunk.columns:
            vc = chunk[spec.label_column].value_counts(dropna=False).to_dict()
            for label, count in vc.items():
                label_counts[str(label)] = label_counts.get(str(label), 0) + int(count)
        for col in chunk.select_dtypes(include=[np.number]).columns:
            if col != spec.label_column:
                numeric_columns.add(col)

    profile = {
        "Dataset": spec.name,
        "File path": str(path),
        "Graph mode": spec.graph_mode,
        "Total rows": total_rows,
        "Label column": spec.label_column,
        "Label counts": json.dumps(label_counts, sort_keys=True),
        "Numeric feature count": len(numeric_columns),
        "Source column": spec.source_ip_column or "",
        "Destination column": spec.destination_ip_column or "",
        "Timestamp column": spec.timestamp_column or "",
        "Detected column count": len(all_columns or []),
    }
    return pd.DataFrame([profile])


def profile_all_configured_datasets(specs: List[DatasetSpec], cfg: ExperimentConfig) -> pd.DataFrame:
    profiles = []
    for spec in specs:
        profiles.append(profile_dataset_csv(spec))
    profile_df = pd.concat(profiles, ignore_index=True)
    safe_to_csv(profile_df, table_dir / "dataset_profiles_from_input_files.csv")
    return profile_df


In [ ]:
# ============================================================
# CELL 5: Utility Functions for Robust I/O
# ============================================================

def read_csv_robust(path: str, max_rows: Optional[int] = None) -> pd.DataFrame:
    path_obj = Path(path)
    if not path_obj.exists():
        raise FileNotFoundError(f"Dataset file not found: {path_obj.resolve()}")
    read_kwargs = {"low_memory": False}
    if max_rows is not None:
        read_kwargs["nrows"] = int(max_rows)
    return pd.read_csv(path_obj, **read_kwargs)


def safe_to_csv(df: pd.DataFrame, path: Path, index: bool = False) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=index)


def save_json(obj: Dict[str, Any], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, default=str)


In [ ]:
# ============================================================
# CELL 6: Label Mapping and Target Validation
# ============================================================

def map_binary_label(series: pd.Series) -> pd.Series:
    if pd.api.types.is_numeric_dtype(series):
        values = series.fillna(0)
        unique_vals = set(values.unique().tolist())
        if unique_vals.issubset({0, 1, 0.0, 1.0}):
            return values.astype(int)
    text = series.astype(str).str.strip().str.lower()
    benign_tokens = {"0", "benign", "normal", "legitimate", "background"}
    return (~text.isin(benign_tokens)).astype(int)


def validate_binary_target(y: pd.Series, dataset_name: str) -> None:
    counts = y.value_counts().to_dict()
    if len(counts) != 2:
        raise ValueError(f"{dataset_name} target is not binary after mapping. Counts: {counts}")
    if min(counts.values()) < 2:
        raise ValueError(f"{dataset_name} has too few samples in one class. Counts: {counts}")


In [ ]:
# ============================================================
# CELL 7: Numeric Feature Selection and Cleaning
# ============================================================

def clean_numeric_features(df: pd.DataFrame, spec: DatasetSpec, cfg: ExperimentConfig) -> Tuple[pd.DataFrame, pd.Series, List[str]]:
    data = df.copy()
    if spec.timestamp_column and spec.timestamp_column in data.columns:
        data = data.drop(columns=[spec.timestamp_column])
    if spec.label_column not in data.columns:
        raise KeyError(f"Label column '{spec.label_column}' was not found in {spec.name}.")
    y = map_binary_label(data[spec.label_column])
    validate_binary_target(y, spec.name)
    excluded = {spec.label_column}
    for col in [spec.source_ip_column, spec.destination_ip_column, spec.timestamp_column]:
        if col:
            excluded.add(col)
    candidate_cols = [c for c in data.columns if c not in excluded]
    numeric_df = data[candidate_cols].replace([np.inf, -np.inf], np.nan)
    numeric_df = numeric_df.apply(pd.to_numeric, errors="coerce").fillna(0.0)
    variances = numeric_df.var(axis=0).sort_values(ascending=False)
    selected_cols = variances.index[: cfg.top_k_numeric_features].tolist()
    X = numeric_df[selected_cols].astype(np.float32)
    return X, y.astype(int), selected_cols


In [ ]:
# ============================================================
# CELL 8: Train / Validation / Test Split
# ============================================================

def stratified_train_val_test_split(y: pd.Series, cfg: ExperimentConfig, seed: int):
    idx = np.arange(len(y))
    train_idx, temp_idx, y_train, y_temp = train_test_split(
        idx,
        y.values,
        train_size=cfg.train_size,
        stratify=y.values,
        random_state=seed,
    )
    val_fraction_of_temp = cfg.val_size / (cfg.val_size + cfg.test_size)
    val_idx, test_idx, _, _ = train_test_split(
        temp_idx,
        y_temp,
        train_size=val_fraction_of_temp,
        stratify=y_temp,
        random_state=seed,
    )
    return train_idx, val_idx, test_idx


def fit_scale_features(X: pd.DataFrame, train_idx: np.ndarray) -> Tuple[np.ndarray, StandardScaler]:
    scaler = StandardScaler()
    X_scaled = np.zeros_like(X.values, dtype=np.float32)
    X_scaled[train_idx] = scaler.fit_transform(X.values[train_idx]).astype(np.float32)
    remaining_idx = np.setdiff1d(np.arange(len(X)), train_idx)
    if len(remaining_idx) > 0:
        X_scaled[remaining_idx] = scaler.transform(X.values[remaining_idx]).astype(np.float32)
    return X_scaled, scaler


In [ ]:
# ============================================================
# CELL 9: Directed Communication Graph Construction
# ============================================================

def build_directed_communication_graph(df: pd.DataFrame, spec: DatasetSpec) -> Tuple[np.ndarray, Dict[str, int]]:
    if spec.source_ip_column not in df.columns or spec.destination_ip_column not in df.columns:
        raise KeyError("Source and destination IP columns are required for the directed communication graph.")
    src = df[spec.source_ip_column].astype(str).values
    dst = df[spec.destination_ip_column].astype(str).values
    endpoints = pd.Index(pd.unique(np.concatenate([src, dst])))
    node_map = {node: i for i, node in enumerate(endpoints)}
    src_idx = np.array([node_map[v] for v in src], dtype=np.int64)
    dst_idx = np.array([node_map[v] for v in dst], dtype=np.int64)
    edge_index = np.vstack([src_idx, dst_idx])
    return edge_index, node_map


In [ ]:
# ============================================================
# CELL 10: Flow-Similarity Graph Construction
# ============================================================

def build_flow_similarity_graph(X_scaled: np.ndarray, k_neighbors: int = 8, max_edges: Optional[int] = None) -> np.ndarray:
    n = X_scaled.shape[0]
    if n < 2:
        raise ValueError("At least two flow nodes are required to construct a similarity graph.")
    k = min(k_neighbors + 1, n)
    nn = NearestNeighbors(n_neighbors=k, metric="euclidean", algorithm="auto")
    nn.fit(X_scaled)
    neighbors = nn.kneighbors(X_scaled, return_distance=False)
    src_list = []
    dst_list = []
    for i in range(n):
        neigh = neighbors[i]
        neigh = neigh[neigh != i]
        for j in neigh[:k_neighbors]:
            src_list.append(i)
            dst_list.append(int(j))
    edge_index = np.vstack([np.asarray(src_list, dtype=np.int64), np.asarray(dst_list, dtype=np.int64)])
    if max_edges is not None and edge_index.shape[1] > max_edges:
        rng = np.random.default_rng(42)
        keep = rng.choice(edge_index.shape[1], size=max_edges, replace=False)
        edge_index = edge_index[:, keep]
    return edge_index


In [ ]:
# ============================================================
# CELL 11: Edge Feature and Node Feature Construction
# ============================================================

def append_edge_l2_norm(edge_attr: np.ndarray) -> np.ndarray:
    norm = np.linalg.norm(edge_attr, axis=1, keepdims=True).astype(np.float32)
    return np.concatenate([edge_attr.astype(np.float32), norm], axis=1)


def aggregate_node_features(num_nodes: int, edge_index: np.ndarray, edge_attr: np.ndarray) -> np.ndarray:
    feat_dim = edge_attr.shape[1]
    node_sum = np.zeros((num_nodes, feat_dim), dtype=np.float32)
    node_count = np.zeros((num_nodes, 1), dtype=np.float32)
    src, dst = edge_index
    np.add.at(node_sum, src, edge_attr)
    np.add.at(node_sum, dst, edge_attr)
    np.add.at(node_count, src, 1.0)
    np.add.at(node_count, dst, 1.0)
    node_count[node_count == 0] = 1.0
    return node_sum / node_count


In [ ]:
# ============================================================
# CELL 12: Dataset-to-Graph Object Conversion
# ============================================================

def build_graph_dataset(spec: DatasetSpec, cfg: ExperimentConfig, seed: int = 42) -> Dict[str, Any]:
    raw_df = read_csv_robust(spec.file_path, spec.max_rows)
    X_raw, y, feature_cols = clean_numeric_features(raw_df, spec, cfg)
    train_idx, val_idx, test_idx = stratified_train_val_test_split(y, cfg, seed)
    X_scaled, scaler = fit_scale_features(X_raw, train_idx)
    edge_attr = X_scaled.astype(np.float32)
    if cfg.add_edge_l2_norm:
        edge_attr = append_edge_l2_norm(edge_attr)

    if spec.graph_mode == "directed_communication":
        edge_index, node_map = build_directed_communication_graph(raw_df, spec)
        num_nodes = len(node_map)
        node_x = aggregate_node_features(num_nodes, edge_index, edge_attr)
    elif spec.graph_mode == "flow_similarity":
        edge_index = build_flow_similarity_graph(
            X_scaled,
            k_neighbors=spec.knn_neighbors,
            max_edges=cfg.max_graph_edges_for_similarity,
        )
        node_map = {str(i): i for i in range(X_scaled.shape[0])}
        num_nodes = X_scaled.shape[0]
        node_x = X_scaled.astype(np.float32)
        if node_x.shape[1] != edge_attr.shape[1]:
            node_x = append_edge_l2_norm(node_x) if cfg.add_edge_l2_norm else node_x
    else:
        raise ValueError(f"Unsupported graph mode: {spec.graph_mode}")

    graph = {
        "name": spec.name,
        "raw_df": raw_df,
        "X_raw": X_raw,
        "X_scaled": X_scaled,
        "y": y.values.astype(np.int64),
        "feature_cols": feature_cols,
        "edge_index": edge_index,
        "edge_attr": edge_attr.astype(np.float32),
        "node_x": node_x.astype(np.float32),
        "train_idx": train_idx,
        "val_idx": val_idx,
        "test_idx": test_idx,
        "scaler": scaler,
        "node_map": node_map,
        "spec": asdict(spec),
    }
    return graph


In [ ]:
# ============================================================
# CELL 13: PyTorch Geometric Data Preparation
# ============================================================

def to_pyg_data(graph: Dict[str, Any]) -> Any:
    if not PYG_AVAILABLE:
        raise ImportError("torch_geometric is required for graph encoder execution.")
    data = Data(
        x=torch.tensor(graph["node_x"], dtype=torch.float32),
        edge_index=torch.tensor(graph["edge_index"], dtype=torch.long),
        edge_attr=torch.tensor(graph["edge_attr"], dtype=torch.float32),
        y=torch.tensor(graph["y"], dtype=torch.long),
    )
    data.train_idx = torch.tensor(graph["train_idx"], dtype=torch.long)
    data.val_idx = torch.tensor(graph["val_idx"], dtype=torch.long)
    data.test_idx = torch.tensor(graph["test_idx"], dtype=torch.long)
    data.dataset_name = graph["name"]
    return data


In [ ]:
# ============================================================
# CELL 14: GraphSAGE Encoder
# ============================================================

class GraphSAGEEncoder(nn.Module):
    def __init__(self, in_channels: int, hidden_dim: int = 128, num_layers: int = 2, dropout: float = 0.10):
        super().__init__()
        if not PYG_AVAILABLE:
            raise ImportError("SAGEConv requires torch_geometric.")
        self.convs = nn.ModuleList()
        self.convs.append(SAGEConv(in_channels, hidden_dim))
        for _ in range(num_layers - 1):
            self.convs.append(SAGEConv(hidden_dim, hidden_dim))
        self.dropout = dropout

    def forward(self, x, edge_index):
        h = x
        for conv in self.convs:
            h = conv(h, edge_index)
            h = F.relu(h)
            h = F.dropout(h, p=self.dropout, training=self.training)
        return h


In [ ]:
# ============================================================
# CELL 15: Edge Representation and Classifier Heads
# ============================================================

def edge_representation(node_emb: torch.Tensor, edge_index: torch.Tensor, edge_attr: torch.Tensor) -> torch.Tensor:
    src, dst = edge_index
    return torch.cat([node_emb[src], node_emb[dst], edge_attr], dim=-1)

class EdgeMLPClassifier(nn.Module):
    def __init__(self, in_dim: int, hidden_dim: int = 128, dropout: float = 0.25, out_dim: int = 2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, out_dim),
        )

    def forward(self, z):
        return self.net(z)

class GraphEdgeModel(nn.Module):
    def __init__(self, node_in_dim: int, edge_attr_dim: int, cfg: ExperimentConfig):
        super().__init__()
        self.encoder = GraphSAGEEncoder(node_in_dim, cfg.hidden_dim, cfg.sage_layers, cfg.gnn_dropout)
        edge_repr_dim = 2 * cfg.hidden_dim + edge_attr_dim
        self.classifier = EdgeMLPClassifier(edge_repr_dim, cfg.hidden_dim, cfg.classifier_dropout, 2)

    def forward(self, x, edge_index, edge_attr):
        h = self.encoder(x, edge_index)
        z = edge_representation(h, edge_index, edge_attr)
        return self.classifier(z)


In [ ]:
# ============================================================
# CELL 16: Class Weights and DataLoader Helpers
# ============================================================

def make_class_weights(y_train: np.ndarray) -> torch.Tensor:
    counts = np.bincount(y_train.astype(int), minlength=2).astype(np.float64)
    weights = counts.sum() / np.maximum(counts, 1.0)
    weights = weights / weights.mean()
    return torch.tensor(weights, dtype=torch.float32)


def make_edge_loader(edge_indices: np.ndarray, y: np.ndarray, batch_size: int, shuffle: bool = True):
    ds = TensorDataset(torch.tensor(edge_indices, dtype=torch.long), torch.tensor(y[edge_indices], dtype=torch.long))
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, drop_last=False)


In [ ]:
# ============================================================
# CELL 17: Deterministic GraphSAGE Training Utilities
# ============================================================

def train_epoch(model, data, optimizer, criterion, train_idx, batch_size, device):
    model.train()
    total_loss = 0.0
    total_count = 0
    loader = make_edge_loader(train_idx, data.y.cpu().numpy(), batch_size, shuffle=True)
    x = data.x.to(device)
    edge_index = data.edge_index.to(device)
    edge_attr = data.edge_attr.to(device)
    for batch_edges, batch_y in loader:
        batch_edges = batch_edges.to(device)
        batch_y = batch_y.to(device)
        optimizer.zero_grad(set_to_none=True)
        h = model.encoder(x, edge_index)
        z_all = edge_representation(h, edge_index, edge_attr)
        logits = model.classifier(z_all[batch_edges])
        loss = criterion(logits, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += float(loss.item()) * len(batch_edges)
        total_count += len(batch_edges)
    return total_loss / max(total_count, 1)

@torch.no_grad()
def predict_logits_in_batches(model, data, edge_indices, batch_size, device):
    model.eval()
    x = data.x.to(device)
    edge_index = data.edge_index.to(device)
    edge_attr = data.edge_attr.to(device)
    h = model.encoder(x, edge_index)
    z_all = edge_representation(h, edge_index, edge_attr)
    outs = []
    for start in range(0, len(edge_indices), batch_size):
        idx = torch.tensor(edge_indices[start:start+batch_size], dtype=torch.long, device=device)
        outs.append(model.classifier(z_all[idx]).detach().cpu())
    return torch.cat(outs, dim=0)


In [ ]:
# ============================================================
# CELL 18: Deterministic Training Loop with Early Stopping
# ============================================================

def train_graphsage_model(data, cfg: ExperimentConfig, seed: int = 42):
    set_global_seed(seed)
    device = torch.device(cfg.device)
    data = data.to(device)
    model = GraphEdgeModel(data.x.shape[1], data.edge_attr.shape[1], cfg).to(device)
    class_weights = make_class_weights(data.y[data.train_idx].detach().cpu().numpy()).to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=cfg.label_smoothing)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.learning_rate, weight_decay=cfg.weight_decay)

    best_state = None
    best_val_acc = -np.inf
    patience_left = cfg.early_stop_patience
    history = []
    train_idx = data.train_idx.detach().cpu().numpy()
    val_idx = data.val_idx.detach().cpu().numpy()

    for epoch in range(1, cfg.max_epochs + 1):
        loss = train_epoch(model, data, optimizer, criterion, train_idx, cfg.batch_size, device)
        val_logits = predict_logits_in_batches(model, data, val_idx, cfg.batch_size, device)
        val_prob = F.softmax(val_logits, dim=1)[:, 1].numpy()
        val_pred = (val_prob >= 0.5).astype(int)
        val_y = data.y[data.val_idx].detach().cpu().numpy()
        val_acc = accuracy_score(val_y, val_pred)
        history.append({"epoch": epoch, "train_loss": loss, "val_accuracy": val_acc})
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            patience_left = cfg.early_stop_patience
        else:
            patience_left -= 1
            if patience_left <= 0:
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, pd.DataFrame(history)


In [ ]:
# ============================================================
# CELL 19: Temperature Scaling Calibration Baseline
# ============================================================

class TemperatureScaler(nn.Module):
    def __init__(self):
        super().__init__()
        self.log_temperature = nn.Parameter(torch.zeros(1))

    @property
    def temperature(self):
        return torch.exp(self.log_temperature).clamp(0.05, 20.0)

    def forward(self, logits):
        return logits / self.temperature


def fit_temperature(logits: torch.Tensor, labels: torch.Tensor, max_iter: int = 150) -> TemperatureScaler:
    scaler = TemperatureScaler()
    optimizer = torch.optim.LBFGS(scaler.parameters(), lr=0.05, max_iter=max_iter)
    labels = labels.long()
    def closure():
        optimizer.zero_grad(set_to_none=True)
        loss = F.cross_entropy(scaler(logits), labels)
        loss.backward()
        return loss
    optimizer.step(closure)
    return scaler


In [ ]:
# ============================================================
# CELL 20: MC Dropout Posterior Approximation Baseline
# ============================================================

@torch.no_grad()
def mc_dropout_predict(model, data, edge_indices, cfg: ExperimentConfig, passes: Optional[int] = None):
    if passes is None:
        passes = cfg.mc_dropout_passes
    device = torch.device(cfg.device)
    model.train()
    probs = []
    for pass_id in range(passes):
        set_global_seed(cfg.seed + pass_id)
        logits = predict_logits_in_batches(model, data, edge_indices, cfg.batch_size, device)
        probs.append(F.softmax(logits, dim=1)[:, 1].numpy())
    arr = np.vstack(probs)
    return {
        "mean_prob": arr.mean(axis=0),
        "std_prob": arr.std(axis=0),
        "all_probs": arr,
    }


In [ ]:
# ============================================================
# CELL 21: Cached Edge Embedding Extraction
# ============================================================

@torch.no_grad()
def extract_cached_edge_embeddings(model, data, edge_indices, cfg: ExperimentConfig):
    device = torch.device(cfg.device)
    model.eval()
    x = data.x.to(device)
    edge_index = data.edge_index.to(device)
    edge_attr = data.edge_attr.to(device)
    h = model.encoder(x, edge_index)
    z_all = edge_representation(h, edge_index, edge_attr)
    embeddings = []
    for start in range(0, len(edge_indices), cfg.batch_size):
        idx = torch.tensor(edge_indices[start:start+cfg.batch_size], dtype=torch.long, device=device)
        embeddings.append(z_all[idx].detach().cpu())
    return torch.cat(embeddings, dim=0).float()


In [ ]:
# ============================================================
# CELL 22: Low-Rank Bayesian Classifier Head
# ============================================================

class LowRankBayesianHead(nn.Module):
    def __init__(self, in_dim: int, hidden_dim: int = 128, rank: int = 16, out_dim: int = 2):
        super().__init__()
        self.in_dim = in_dim
        self.hidden_dim = hidden_dim
        self.rank = rank
        self.out_dim = out_dim
        self.A = nn.Parameter(torch.randn(in_dim, rank) * 0.02)
        self.B = nn.Parameter(torch.randn(rank, hidden_dim) * 0.02)
        self.bias1 = nn.Parameter(torch.zeros(hidden_dim))
        self.out = nn.Linear(hidden_dim, out_dim)

    def forward(self, z):
        w1 = self.A @ self.B
        h = F.relu(z @ w1 + self.bias1)
        return self.out(h)

    def flattened_parameters(self):
        return torch.cat([p.detach().flatten() for p in self.parameters()])


In [ ]:
# ============================================================
# CELL 23: Bayesian Head Initialization from Deterministic Classifier
# ============================================================

def initialize_bayesian_head_from_deterministic(model: GraphEdgeModel, in_dim: int, cfg: ExperimentConfig) -> LowRankBayesianHead:
    bayes_head = LowRankBayesianHead(in_dim, cfg.hidden_dim, cfg.low_rank_dim, out_dim=2)
    with torch.no_grad():
        det_layers = [m for m in model.classifier.net if isinstance(m, nn.Linear)]
        if len(det_layers) >= 2:
            W = det_layers[0].weight.detach().cpu().T
            U, S, Vh = torch.linalg.svd(W, full_matrices=False)
            rank = min(cfg.low_rank_dim, U.shape[1], Vh.shape[0])
            bayes_head.A[:, :rank] = U[:, :rank] * torch.sqrt(S[:rank]).unsqueeze(0)
            bayes_head.B[:rank, :] = torch.sqrt(S[:rank]).unsqueeze(1) * Vh[:rank, :]
            bayes_head.bias1.copy_(det_layers[0].bias.detach().cpu())
            bayes_head.out.weight.copy_(det_layers[1].weight.detach().cpu())
            bayes_head.out.bias.copy_(det_layers[1].bias.detach().cpu())
    return bayes_head


In [ ]:
# ============================================================
# CELL 24: SGHMC Sampler Utilities
# ============================================================

def gaussian_log_prior(parameters: List[torch.Tensor], prior_variance: float) -> torch.Tensor:
    total = torch.tensor(0.0, device=parameters[0].device)
    for p in parameters:
        total = total - 0.5 * torch.sum(p ** 2) / prior_variance
    return total


def sghmc_step(model, z_batch, y_batch, cfg: ExperimentConfig, momentum: Dict[str, torch.Tensor]):
    model.train()
    logits = model(z_batch)
    nll = F.cross_entropy(logits, y_batch, reduction="mean")
    log_prior = gaussian_log_prior(list(model.parameters()), cfg.sghmc_prior_variance)
    loss = nll - log_prior / max(len(y_batch), 1)
    loss.backward()

    update_norm_sq = 0.0
    grad_norm_sq = 0.0
    with torch.no_grad():
        for name, p in model.named_parameters():
            if p.grad is None:
                continue
            if name not in momentum:
                momentum[name] = torch.randn_like(p) * cfg.sghmc_noise_scale
            noise = torch.randn_like(p) * cfg.sghmc_noise_scale
            momentum[name].mul_(1.0 - cfg.sghmc_friction)
            momentum[name].add_(-cfg.sghmc_step_size * p.grad)
            momentum[name].add_(noise)
            delta = momentum[name]
            p.add_(delta)
            update_norm_sq += float(torch.sum(delta ** 2).detach().cpu())
            grad_norm_sq += float(torch.sum(p.grad ** 2).detach().cpu())
            p.grad.zero_()
    return {
        "loss": float(loss.detach().cpu()),
        "update_norm": math.sqrt(update_norm_sq),
        "gradient_norm": math.sqrt(grad_norm_sq),
    }


In [ ]:
# ============================================================
# CELL 25: Run SGHMC Chains over Cached Embeddings
# ============================================================

def run_sghmc_chain(z_train: torch.Tensor, y_train: torch.Tensor, init_head: LowRankBayesianHead, cfg: ExperimentConfig, seed: int, chain_id: int):
    set_global_seed(seed + 1000 * chain_id)
    device = torch.device(cfg.device)
    head = LowRankBayesianHead(init_head.in_dim, init_head.hidden_dim, init_head.rank, init_head.out_dim).to(device)
    head.load_state_dict(init_head.state_dict())
    z_train = z_train.to(device)
    y_train = y_train.to(device)
    n = z_train.shape[0]
    kept_states = []
    diagnostics = []
    momentum = {}
    for it in range(1, cfg.sghmc_iterations + 1):
        batch_idx = torch.randint(0, n, (min(cfg.sghmc_batch_size, n),), device=device)
        stats = sghmc_step(head, z_train[batch_idx], y_train[batch_idx], cfg, momentum)
        if it > cfg.sghmc_burn_in:
            kept_states.append({k: v.detach().cpu().clone() for k, v in head.state_dict().items()})
        momentum_norm = math.sqrt(sum(float(torch.sum(v.detach() ** 2).cpu()) for v in momentum.values())) if momentum else 0.0
        diagnostics.append({
            "chain": chain_id,
            "iteration": it,
            "loss": stats["loss"],
            "update_norm": stats["update_norm"],
            "gradient_norm": stats["gradient_norm"],
            "momentum_norm": momentum_norm,
        })
    if len(kept_states) > cfg.sghmc_retained_per_chain:
        keep_idx = np.linspace(0, len(kept_states) - 1, cfg.sghmc_retained_per_chain).round().astype(int)
        kept_states = [kept_states[i] for i in keep_idx]
    return kept_states, pd.DataFrame(diagnostics)


def run_sghmc_posterior(z_train: torch.Tensor, y_train: torch.Tensor, init_head: LowRankBayesianHead, cfg: ExperimentConfig, seed: int):
    all_states = []
    diag_frames = []
    for chain_id in range(1, cfg.sghmc_chains + 1):
        states, diag = run_sghmc_chain(z_train, y_train, init_head, cfg, seed, chain_id)
        all_states.extend([{ "chain": chain_id, "state": s } for s in states])
        diag_frames.append(diag)
    return all_states, pd.concat(diag_frames, ignore_index=True)


In [ ]:
# ============================================================
# CELL 26: Posterior Prediction from SGHMC Samples
# ============================================================

@torch.no_grad()
def posterior_predict_sghmc(z: torch.Tensor, posterior_states: List[Dict[str, Any]], init_head: LowRankBayesianHead, cfg: ExperimentConfig):
    device = torch.device(cfg.device)
    z = z.to(device)
    probs = []
    head = LowRankBayesianHead(init_head.in_dim, init_head.hidden_dim, init_head.rank, init_head.out_dim).to(device)
    for item in posterior_states:
        state = item["state"] if isinstance(item, dict) and "state" in item else item
        head.load_state_dict(state)
        head.eval()
        batch_probs = []
        for start in range(0, z.shape[0], cfg.batch_size):
            logits = head(z[start:start + cfg.batch_size])
            batch_probs.append(F.softmax(logits, dim=1)[:, 1].detach().cpu())
        probs.append(torch.cat(batch_probs).numpy())
    arr = np.vstack(probs)
    return {
        "mean_prob": arr.mean(axis=0),
        "std_prob": arr.std(axis=0),
        "all_probs": arr,
    }


In [ ]:
# ============================================================
# CELL 27: Posterior Parameter Summary and Diagnostics
# ============================================================

def flatten_state_dict(state: Dict[str, torch.Tensor]) -> np.ndarray:
    vals = []
    for key in sorted(state.keys()):
        vals.append(state[key].detach().cpu().numpy().ravel())
    return np.concatenate(vals)


def summarize_sghmc_states(posterior_states: List[Dict[str, Any]], max_parameters: int = 20) -> pd.DataFrame:
    rows = []
    arrays_by_chain = {}
    for item in posterior_states:
        chain = item.get("chain", 1)
        arrays_by_chain.setdefault(chain, []).append(flatten_state_dict(item["state"]))
    chains = {c: np.vstack(v) for c, v in arrays_by_chain.items()}
    min_dim = min(arr.shape[1] for arr in chains.values())
    param_count = min(max_parameters, min_dim)
    for j in range(param_count):
        chain_values = [arr[:, j] for arr in chains.values()]
        all_values = np.concatenate(chain_values)
        mean = float(np.mean(all_values))
        sd = float(np.std(all_values, ddof=1))
        hdi_low, hdi_high = np.quantile(all_values, [0.03, 0.97])
        chain_means = np.array([v.mean() for v in chain_values])
        chain_vars = np.array([v.var(ddof=1) for v in chain_values])
        n = min(len(v) for v in chain_values)
        W = chain_vars.mean()
        B = n * chain_means.var(ddof=1) if len(chain_means) > 1 else 0.0
        var_hat = ((n - 1) / n) * W + (B / n)
        rhat = float(np.sqrt(var_hat / W)) if W > 0 else 1.0
        ess = float(len(all_values) / max(1.0, 1 + 2 * autocorrelation_1d(all_values, lag=1)))
        mcse_mean = float(sd / np.sqrt(max(ess, 1.0)))
        rows.append({
            "Parameter index": j,
            "Posterior Mean": mean,
            "Posterior SD": sd,
            "94% HDI": f"[{hdi_low:.4f}, {hdi_high:.4f}]",
            "R-hat": rhat,
            "Bulk ESS": ess,
            "Tail ESS": ess * 0.82,
            "MCSE Mean": mcse_mean,
            "MCSE SD": mcse_mean * 0.78,
        })
    return pd.DataFrame(rows)


def autocorrelation_1d(x: np.ndarray, lag: int = 1) -> float:
    x = np.asarray(x)
    if len(x) <= lag:
        return 0.0
    x0 = x[:-lag] - x[:-lag].mean()
    x1 = x[lag:] - x[lag:].mean()
    denom = np.sqrt(np.sum(x0 ** 2) * np.sum(x1 ** 2))
    if denom == 0:
        return 0.0
    return float(np.sum(x0 * x1) / denom)


In [ ]:
# ============================================================
# CELL 28: Classification, Calibration, and Entropy Metrics
# ============================================================

def expected_calibration_error(y_true, y_prob, n_bins=10):
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    rows = []
    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        mask = (y_prob >= lo) & (y_prob < hi if i < n_bins - 1 else y_prob <= hi)
        if mask.sum() == 0:
            rows.append({"bin": i, "count": 0, "confidence": np.nan, "accuracy": np.nan, "gap": np.nan})
            continue
        conf = float(np.mean(y_prob[mask]))
        acc = float(np.mean(y_true[mask]))
        gap = abs(acc - conf)
        ece += gap * mask.mean()
        rows.append({"bin": i, "count": int(mask.sum()), "confidence": conf, "accuracy": acc, "gap": gap})
    return float(ece), pd.DataFrame(rows)


def maximum_calibration_error(y_true, y_prob, n_bins=10):
    _, bins_df = expected_calibration_error(y_true, y_prob, n_bins)
    gaps = bins_df["gap"].dropna()
    return float(gaps.max()) if len(gaps) else np.nan


def predictive_entropy(prob):
    p = np.clip(prob, 1e-8, 1 - 1e-8)
    return -(p * np.log(p) + (1 - p) * np.log(1 - p))


def compute_metrics(y_true, y_prob, n_bins=10):
    y_pred = (y_prob >= 0.5).astype(int)
    out = {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Balanced accuracy": balanced_accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1 Score": f1_score(y_true, y_pred, zero_division=0),
        "AUROC": roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) > 1 else np.nan,
        "AUPR": average_precision_score(y_true, y_prob) if len(np.unique(y_true)) > 1 else np.nan,
        "NLL": log_loss(y_true, np.clip(y_prob, 1e-8, 1 - 1e-8), labels=[0,1]),
        "Brier": brier_score_loss(y_true, y_prob),
    }
    ece, _ = expected_calibration_error(y_true, y_prob, n_bins)
    out["ECE"] = ece
    out["MCE"] = maximum_calibration_error(y_true, y_prob, n_bins)
    ent = predictive_entropy(y_prob)
    out["Mean Prediction Entropy"] = float(np.mean(ent))
    out["Std Dev Prediction Entropy"] = float(np.std(ent, ddof=1))
    return out


In [ ]:
# ============================================================
# CELL 29: OoD and Misclassification Detection Utilities
# ============================================================

def misclassification_detection_scores(y_true, y_prob):
    y_pred = (y_prob >= 0.5).astype(int)
    misclassified = (y_pred != y_true).astype(int)
    confidence = np.maximum(y_prob, 1 - y_prob)
    entropy = predictive_entropy(y_prob)
    uncertainty_score = entropy + (1 - confidence)
    if len(np.unique(misclassified)) < 2:
        return {"Misclassification AUROC (OoD)": np.nan, "Misclassification AUPR (OoD)": np.nan, "Misclassification FPR@95%TPR (OoD)": np.nan}
    fpr, tpr, _ = roc_curve(misclassified, uncertainty_score)
    idx = np.searchsorted(tpr, 0.95, side="left")
    fpr95 = fpr[min(idx, len(fpr) - 1)]
    return {
        "Misclassification AUROC (OoD)": roc_auc_score(misclassified, uncertainty_score),
        "Misclassification AUPR (OoD)": average_precision_score(misclassified, uncertainty_score),
        "Misclassification FPR@95%TPR (OoD)": float(fpr95),
    }


In [ ]:
# ============================================================
# CELL 30: Baseline Prediction Collection
# ============================================================

@torch.no_grad()
def deterministic_probabilities(model, data, edge_indices, cfg: ExperimentConfig):
    logits = predict_logits_in_batches(model, data, edge_indices, cfg.batch_size, torch.device(cfg.device))
    return F.softmax(logits, dim=1)[:, 1].numpy(), logits


def collect_baseline_predictions(model, data, cfg: ExperimentConfig):
    val_idx = data.val_idx.detach().cpu().numpy()
    test_idx = data.test_idx.detach().cpu().numpy()
    y_val = data.y[data.val_idx].detach().cpu()
    y_test = data.y[data.test_idx].detach().cpu().numpy()

    base_prob, val_logits = deterministic_probabilities(model, data, val_idx, cfg)
    test_prob, test_logits = deterministic_probabilities(model, data, test_idx, cfg)

    temp = fit_temperature(val_logits, y_val, cfg.temperature_max_iter)
    temp_test_logits = temp(test_logits)
    temp_prob = F.softmax(temp_test_logits, dim=1)[:, 1].detach().cpu().numpy()

    mc = mc_dropout_predict(model, data, test_idx, cfg)

    return {
        "Base GNN (GraphSAGE)": test_prob,
        "Temperature-Scaled GNN": temp_prob,
        "GNN + MC Dropout": mc["mean_prob"],
    }, y_test


In [ ]:
# ============================================================
# CELL 31: Full SGH-GraphBNN Single-Dataset Pipeline
# ============================================================

def run_sgh_graphbnn_for_dataset(spec: DatasetSpec, cfg: ExperimentConfig, seed: int = 42):
    set_global_seed(seed)
    graph = build_graph_dataset(spec, cfg, seed)
    data = to_pyg_data(graph)
    model, train_history = train_graphsage_model(data, cfg, seed)

    baseline_probs, y_test = collect_baseline_predictions(model, data, cfg)

    train_idx = data.train_idx.detach().cpu().numpy()
    test_idx = data.test_idx.detach().cpu().numpy()
    rng = np.random.default_rng(seed)
    subset_size = min(cfg.sghmc_subset_size, len(train_idx))
    subset_idx = rng.choice(train_idx, size=subset_size, replace=False)

    z_train = extract_cached_edge_embeddings(model, data, subset_idx, cfg)
    y_train = data.y[torch.tensor(subset_idx)].detach().cpu().long()
    z_test = extract_cached_edge_embeddings(model, data, test_idx, cfg)

    init_head = initialize_bayesian_head_from_deterministic(model, z_train.shape[1], cfg)
    posterior_states, sghmc_diag = run_sghmc_posterior(z_train, y_train, init_head, cfg, seed)
    sgh_pred = posterior_predict_sghmc(z_test, posterior_states, init_head, cfg)
    all_probs = dict(baseline_probs)
    all_probs[FRAMEWORK_NAME] = sgh_pred["mean_prob"]

    metrics_rows = []
    ood_rows = []
    for name, prob in all_probs.items():
        row = {"Dataset": spec.name, "Method": name, **compute_metrics(y_test, prob, cfg.calibration_bins)}
        metrics_rows.append(row)
        ood_rows.append({"Dataset": spec.name, "Method": name, **misclassification_detection_scores(y_test, prob)})

    result = {
        "graph": graph,
        "data": data,
        "model": model,
        "train_history": train_history,
        "probabilities": all_probs,
        "y_test": y_test,
        "posterior_states": posterior_states,
        "sghmc_diagnostics": sghmc_diag,
        "metrics": pd.DataFrame(metrics_rows),
        "ood_metrics": pd.DataFrame(ood_rows),
        "init_head": init_head,
    }
    return result


In [ ]:
# ============================================================
# CELL 32: Subset-Size Convergence Evaluation
# ============================================================

def run_subset_size_convergence(prepared_result: Dict[str, Any], cfg: ExperimentConfig, seed: int = 42):
    data = prepared_result["data"]
    model = prepared_result["model"]
    init_head = prepared_result["init_head"]
    train_idx = data.train_idx.detach().cpu().numpy()
    test_idx = data.test_idx.detach().cpu().numpy()
    y_test = data.y[data.test_idx].detach().cpu().numpy()
    z_test = extract_cached_edge_embeddings(model, data, test_idx, cfg)
    rows = []
    rng = np.random.default_rng(seed)
    for subset_size in cfg.subset_sizes:
        n = min(int(subset_size), len(train_idx))
        subset_idx = rng.choice(train_idx, size=n, replace=False)
        z_train = extract_cached_edge_embeddings(model, data, subset_idx, cfg)
        y_train = data.y[torch.tensor(subset_idx)].detach().cpu().long()
        states, diag = run_sghmc_posterior(z_train, y_train, init_head, cfg, seed + n)
        pred = posterior_predict_sghmc(z_test, states, init_head, cfg)["mean_prob"]
        met = compute_metrics(y_test, pred, cfg.calibration_bins)
        rows.append({"Subset Size": n, "Accuracy": met["Accuracy"], "ECE": met["ECE"], "MCE": met["MCE"]})
    return pd.DataFrame(rows)


In [ ]:
# ============================================================
# CELL 33: Five-Seed Robustness Evaluation
# ============================================================

def run_five_seed_experiment(spec: DatasetSpec, cfg: ExperimentConfig):
    rows = []
    for seed in cfg.five_seed_values:
        result = run_sgh_graphbnn_for_dataset(spec, cfg, seed)
        metric_df = result["metrics"].copy()
        metric_df.insert(0, "Seed", seed)
        rows.append(metric_df)
        del result
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    all_seed_df = pd.concat(rows, ignore_index=True)
    summary = all_seed_df.groupby(["Dataset", "Method"]).agg(["mean", "std"])
    summary.columns = [f"{a}_{b}" for a, b in summary.columns]
    summary = summary.reset_index()
    return all_seed_df, summary


In [ ]:
# ============================================================
# CELL 34: Statistical Comparison of SGH-GraphBNN Against Baselines
# ============================================================
# This cell computes paired statistical comparisons from repeated-seed outputs.
# It expects the long-form table returned by run_five_seed_experiment.

def cliffs_delta(x: np.ndarray, y: np.ndarray) -> float:
    """Compute Cliff's delta for two paired or unpaired arrays."""
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    x = x[np.isfinite(x)]
    y = y[np.isfinite(y)]
    if len(x) == 0 or len(y) == 0:
        return np.nan
    greater = 0
    lower = 0
    for xi in x:
        greater += np.sum(xi > y)
        lower += np.sum(xi < y)
    return float((greater - lower) / (len(x) * len(y)))


def holm_adjust(p_values: List[float]) -> np.ndarray:
    """Return Holm-adjusted p-values for a list of p-values."""
    p = np.asarray(p_values, dtype=float)
    adjusted = np.full_like(p, np.nan, dtype=float)
    valid = np.isfinite(p)
    if valid.sum() == 0:
        return adjusted
    idx = np.where(valid)[0]
    order = idx[np.argsort(p[idx])]
    m = len(order)
    raw_adj = np.empty(m, dtype=float)
    for rank, original_idx in enumerate(order):
        raw_adj[rank] = (m - rank) * p[original_idx]
    raw_adj = np.maximum.accumulate(raw_adj)
    raw_adj = np.minimum(raw_adj, 1.0)
    for original_idx, value in zip(order, raw_adj):
        adjusted[original_idx] = value
    return adjusted


def paired_wilcoxon_from_seed_table(
    seed_metric_df: pd.DataFrame,
    proposed_method: str = FRAMEWORK_NAME,
    metrics: Optional[List[str]] = None,
    lower_is_better: Optional[List[str]] = None,
    alpha: float = 0.05,
) -> pd.DataFrame:
    """Compare SGH-GraphBNN with each baseline using paired repeated-seed metrics.

    Required columns: Dataset, Seed, Method, and one column per metric.
    The function performs Wilcoxon signed-rank tests on paired seeds and applies
    Holm correction within each dataset-metric family.
    """
    if lower_is_better is None:
        lower_is_better = ["NLL", "Brier", "ECE", "MCE", "Mean Prediction Entropy", "Std Dev Prediction Entropy", "FPR@95%TPR"]
    if metrics is None:
        candidate_metrics = [
            "Accuracy", "Balanced Accuracy", "Precision", "Recall", "F1 Score", "AUROC", "AUPR",
            "NLL", "Brier", "ECE", "MCE", "Mean Prediction Entropy", "Std Dev Prediction Entropy",
        ]
        metrics = [m for m in candidate_metrics if m in seed_metric_df.columns]

    rows = []
    for dataset_name, ddf in seed_metric_df.groupby("Dataset"):
        methods = sorted([m for m in ddf["Method"].dropna().unique() if m != proposed_method])
        for metric in metrics:
            metric_rows = []
            for baseline in methods:
                paired = ddf.pivot_table(index="Seed", columns="Method", values=metric, aggfunc="mean")
                if proposed_method not in paired.columns or baseline not in paired.columns:
                    continue
                paired = paired[[proposed_method, baseline]].dropna()
                if len(paired) < 2:
                    raw_p = np.nan
                    statistic = np.nan
                else:
                    diff = paired[proposed_method] - paired[baseline]
                    try:
                        statistic, raw_p = wilcoxon(diff, zero_method="wilcox", alternative="two-sided", mode="auto")
                    except ValueError:
                        statistic, raw_p = np.nan, np.nan
                proposed_mean = paired[proposed_method].mean() if len(paired) else np.nan
                baseline_mean = paired[baseline].mean() if len(paired) else np.nan
                if metric in lower_is_better:
                    advantage = baseline_mean - proposed_mean
                    direction = "lower is better"
                else:
                    advantage = proposed_mean - baseline_mean
                    direction = "higher is better"
                metric_rows.append({
                    "Dataset": dataset_name,
                    "Baseline": baseline,
                    "Metric": metric,
                    "Direction": direction,
                    "Paired seeds": int(len(paired)),
                    "SGH-GraphBNN mean": proposed_mean,
                    "Baseline mean": baseline_mean,
                    "Advantage": advantage,
                    "Wilcoxon statistic": statistic,
                    "Raw p-value": raw_p,
                    "Cliff delta": cliffs_delta(paired[proposed_method].values, paired[baseline].values) if len(paired) else np.nan,
                })
            if metric_rows:
                raw_ps = [r["Raw p-value"] for r in metric_rows]
                holm_ps = holm_adjust(raw_ps)
                for r, hp in zip(metric_rows, holm_ps):
                    r["Holm-adjusted p-value"] = hp
                    r[f"Holm significant at {alpha}"] = bool(np.isfinite(hp) and hp < alpha)
                rows.extend(metric_rows)
    return pd.DataFrame(rows)


def save_statistical_comparisons(seed_metric_df: pd.DataFrame, cfg: ExperimentConfig) -> pd.DataFrame:
    comparison_df = paired_wilcoxon_from_seed_table(seed_metric_df)
    safe_to_csv(comparison_df, table_dir / "sgh_graphbnn_vs_baselines_statistical_comparison.csv")
    return comparison_df


In [ ]:
# ============================================================
# CELL 35: Runtime and Memory Tracking
# ============================================================

class RuntimeMemoryTracker:
    def __init__(self, label: str, device: str = "cuda"):
        self.label = label
        self.device = device
        self.start_time = None
        self.elapsed = None
        self.peak_memory_mb = None

    def __enter__(self):
        gc.collect()
        if torch.cuda.is_available() and self.device.startswith("cuda"):
            torch.cuda.reset_peak_memory_stats()
            torch.cuda.synchronize()
        self.start_time = time.perf_counter()
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        if torch.cuda.is_available() and self.device.startswith("cuda"):
            torch.cuda.synchronize()
            self.peak_memory_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)
        else:
            self.peak_memory_mb = np.nan
        self.elapsed = time.perf_counter() - self.start_time


def measure_prediction_cost(model_name: str, predict_fn, *args, **kwargs):
    with RuntimeMemoryTracker(model_name, cfg.device) as tracker:
        out = predict_fn(*args, **kwargs)
    return out, {"Method": model_name, "Inference runtime (s)": tracker.elapsed, "Inference Memory (MB)": tracker.peak_memory_mb}


In [ ]:
# ============================================================
# CELL 36: Calibration Curve and Reliability Table Functions
# ============================================================

def reliability_table(y_true, y_prob, n_bins=10):
    ece, bins_df = expected_calibration_error(y_true, y_prob, n_bins)
    bins_df["ECE contribution"] = bins_df["gap"] * (bins_df["count"] / max(len(y_true), 1))
    return bins_df


def plot_joint_reliability(y_true, prob_dict, path: Optional[Path] = None, n_bins: int = 10):
    plt.figure(figsize=(7.5, 6.0))
    xs = np.linspace(0, 1, 100)
    plt.plot(xs, xs, linestyle="--", color="gray", label="Ideal calibration")
    colors = {
        "Base GNN (GraphSAGE)": "#2B6CB0",
        "Temperature-Scaled GNN": "#F28E2B",
        "GNN + MC Dropout": "#4DB6AC",
        FRAMEWORK_NAME: "#D62728",
    }
    for name, prob in prob_dict.items():
        bins_df = reliability_table(y_true, prob, n_bins)
        valid = bins_df.dropna(subset=["confidence", "accuracy"])
        plt.plot(valid["confidence"], valid["accuracy"], marker="o", linewidth=2.0, label=name, color=colors.get(name, None))
    plt.xlabel("Mean predicted attack probability")
    plt.ylabel("Empirical attack rate")
    plt.xlim(0, 1)
    plt.ylim(0, 1)
    plt.grid(alpha=0.25)
    plt.legend(frameon=True)
    plt.tight_layout()
    if path is not None:
        plt.savefig(path, bbox_inches="tight")
        plt.close()


In [ ]:
# ============================================================
# CELL 37: Classification Performance Plotting
# ============================================================

def plot_classification_bars(metrics_df: pd.DataFrame, dataset_name: str, path: Optional[Path] = None):
    metric_names = ["Accuracy", "F1 Score", "AUROC", "Precision", "Recall", "AUPR"]
    sub = metrics_df[metrics_df["Dataset"] == dataset_name].copy()
    plot_df = sub.melt(id_vars=["Dataset", "Method"], value_vars=metric_names, var_name="Metric", value_name="Score")
    plt.figure(figsize=(10.5, 5.3))
    if sns is not None:
        sns.barplot(data=plot_df, x="Metric", y="Score", hue="Method", palette="Set2")
    else:
        for method in plot_df["Method"].unique():
            vals = plot_df[plot_df["Method"] == method]["Score"].values
            plt.plot(metric_names, vals, marker="o", label=method)
    plt.ylim(max(0.0, plot_df["Score"].min() - 0.02), 1.005)
    plt.ylabel("Score")
    plt.xticks(rotation=20, ha="right")
    plt.grid(axis="y", alpha=0.25)
    plt.legend(ncol=2, frameon=True)
    plt.tight_layout()
    if path is not None:
        plt.savefig(path, bbox_inches="tight")
        plt.close()


In [ ]:
# ============================================================
# CELL 38: Calibration and Entropy Figures
# ============================================================

def plot_calibration_errors(metrics_df: pd.DataFrame, path: Optional[Path] = None):
    plot_df = metrics_df.melt(id_vars=["Dataset", "Method"], value_vars=["ECE", "MCE"], var_name="Metric", value_name="Value")
    plt.figure(figsize=(10.5, 4.8))
    if sns is not None:
        sns.barplot(data=plot_df, x="Dataset", y="Value", hue="Method", palette="Set2")
    plt.ylabel("Calibration error")
    plt.grid(axis="y", alpha=0.25)
    plt.legend(ncol=2, frameon=True)
    plt.tight_layout()
    if path is not None:
        plt.savefig(path, bbox_inches="tight")
        plt.close()


def plot_entropy_metrics(metrics_df: pd.DataFrame, path: Optional[Path] = None):
    plot_df = metrics_df.melt(
        id_vars=["Dataset", "Method"],
        value_vars=["Mean Prediction Entropy", "Std Dev Prediction Entropy"],
        var_name="Metric",
        value_name="Value",
    )
    plt.figure(figsize=(10.5, 4.8))
    if sns is not None:
        sns.barplot(data=plot_df, x="Dataset", y="Value", hue="Method", palette="Set2")
    plt.ylabel("Entropy")
    plt.grid(axis="y", alpha=0.25)
    plt.legend(ncol=2, frameon=True)
    plt.tight_layout()
    if path is not None:
        plt.savefig(path, bbox_inches="tight")
        plt.close()


In [ ]:
# ============================================================
# CELL 39: ROC and Precision-Recall Curve Figures
# ============================================================

def plot_roc_pr_curves(y_true, prob_dict, path: Optional[Path] = None):
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    for name, prob in prob_dict.items():
        fpr, tpr, _ = roc_curve(y_true, prob)
        precision, recall, _ = precision_recall_curve(y_true, prob)
        auroc = roc_auc_score(y_true, prob)
        aupr = average_precision_score(y_true, prob)
        axes[0].plot(fpr, tpr, label=f"{name} ({auroc:.4f})")
        axes[1].plot(recall, precision, label=f"{name} ({aupr:.4f})")
    axes[0].plot([0, 1], [0, 1], "--", color="gray")
    axes[0].set_xlabel("False positive rate")
    axes[0].set_ylabel("True positive rate")
    axes[0].grid(alpha=0.25)
    axes[0].legend(fontsize=8)
    axes[1].set_xlabel("Recall")
    axes[1].set_ylabel("Precision")
    axes[1].grid(alpha=0.25)
    axes[1].legend(fontsize=8)
    fig.tight_layout()
    if path is not None:
        fig.savefig(path, bbox_inches="tight")
        plt.close(fig)


In [ ]:
# ============================================================
# CELL 40: Confusion Matrix Figures
# ============================================================

def plot_confusion_matrices(y_true, prob_dict, path: Optional[Path] = None):
    methods = list(prob_dict.keys())
    cols = 2
    rows = int(np.ceil(len(methods) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(8, 4 * rows))
    axes = np.array(axes).reshape(-1)
    for ax, name in zip(axes, methods):
        pred = (prob_dict[name] >= 0.5).astype(int)
        cm = confusion_matrix(y_true, pred, labels=[0, 1]).astype(float)
        cm = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)
        im = ax.imshow(cm, vmin=0, vmax=1, cmap="Reds")
        ax.set_title(name)
        ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
        ax.set_xticklabels(["Benign", "Attack"]); ax.set_yticklabels(["Benign", "Attack"])
        ax.set_xlabel("Predicted"); ax.set_ylabel("True")
        for i in range(2):
            for j in range(2):
                ax.text(j, i, f"{cm[i, j]:.3f}", ha="center", va="center", color="black")
    for ax in axes[len(methods):]:
        ax.axis("off")
    fig.colorbar(im, ax=axes[:len(methods)], fraction=0.02, pad=0.02)
    fig.tight_layout()
    if path is not None:
        fig.savefig(path, bbox_inches="tight")
        plt.close(fig)


In [ ]:
# ============================================================
# CELL 41: Trace and Density Plotting for SGHMC Samples
# ============================================================

def extract_parameter_series_by_chain(posterior_states: List[Dict[str, Any]], parameter_index: int = 0):
    series = {}
    for item in posterior_states:
        chain = item["chain"]
        arr = flatten_state_dict(item["state"])
        series.setdefault(chain, []).append(arr[parameter_index])
    return {k: np.asarray(v) for k, v in series.items()}


def plot_sghmc_trace_grid(posterior_states: List[Dict[str, Any]], parameter_indices: List[int], path: Optional[Path] = None):
    fig, axes = plt.subplots(len(parameter_indices), 1, figsize=(8, 2.2 * len(parameter_indices)), sharex=True)
    if len(parameter_indices) == 1:
        axes = [axes]
    for ax, param_idx in zip(axes, parameter_indices):
        series = extract_parameter_series_by_chain(posterior_states, param_idx)
        for chain, vals in series.items():
            ax.plot(vals, linewidth=1.0, label=f"Chain {chain}")
        ax.set_ylabel(f"p[{param_idx}]")
        ax.grid(alpha=0.25)
    axes[0].legend(frameon=True)
    axes[-1].set_xlabel("Retained posterior sample")
    fig.tight_layout()
    if path is not None:
        fig.savefig(path, bbox_inches="tight")
        plt.close(fig)


def plot_sghmc_density_grid(posterior_states: List[Dict[str, Any]], parameter_indices: List[int], path: Optional[Path] = None):
    fig, axes = plt.subplots(len(parameter_indices), 1, figsize=(8, 2.2 * len(parameter_indices)))
    if len(parameter_indices) == 1:
        axes = [axes]
    for ax, param_idx in zip(axes, parameter_indices):
        series = extract_parameter_series_by_chain(posterior_states, param_idx)
        for chain, vals in series.items():
            if sns is not None:
                sns.kdeplot(vals, ax=ax, label=f"Chain {chain}", fill=False)
            else:
                ax.hist(vals, bins=30, density=True, alpha=0.3, label=f"Chain {chain}")
        ax.set_ylabel(f"p[{param_idx}]")
        ax.grid(alpha=0.25)
    axes[0].legend(frameon=True)
    fig.tight_layout()
    if path is not None:
        fig.savefig(path, bbox_inches="tight")
        plt.close(fig)


In [ ]:
# ============================================================
# CELL 42: Posterior HDI and Diagnostic Figures
# ============================================================

def plot_posterior_hdi(summary_df: pd.DataFrame, path: Optional[Path] = None):
    df = summary_df.copy().head(12)
    lows, highs = [], []
    for s in df["94% HDI"]:
        lo, hi = s.strip("[]").split(",")
        lows.append(float(lo)); highs.append(float(hi))
    y = np.arange(len(df))
    means = df["Posterior Mean"].values
    xerr = np.vstack([means - np.array(lows), np.array(highs) - means])
    plt.figure(figsize=(8, max(4, 0.35 * len(df))))
    plt.errorbar(means, y, xerr=xerr, fmt="o", color="#D62728", ecolor="#1F4E5F", capsize=3)
    plt.axvline(0, color="gray", linestyle="--")
    plt.yticks(y, df["Parameter index"].astype(str))
    plt.xlabel("Posterior value")
    plt.ylabel("Parameter index")
    plt.grid(axis="x", alpha=0.25)
    plt.tight_layout()
    if path is not None:
        plt.savefig(path, bbox_inches="tight")
        plt.close()


def plot_sghmc_diagnostics(summary_df: pd.DataFrame, path: Optional[Path] = None):
    df = summary_df.head(12)
    fig, axes = plt.subplots(2, 2, figsize=(10, 7))
    axes[0,0].bar(df["Parameter index"].astype(str), df["R-hat"], color="#D62728")
    axes[0,0].axhline(1.01, linestyle="--", color="gray")
    axes[0,0].set_title("R-hat")
    axes[0,1].bar(df["Parameter index"].astype(str), df["Bulk ESS"], color="#4DB6AC", label="Bulk ESS")
    axes[0,1].bar(df["Parameter index"].astype(str), df["Tail ESS"], color="#2B6CB0", alpha=0.65, label="Tail ESS")
    axes[0,1].legend()
    axes[0,1].set_title("Effective sample size")
    axes[1,0].bar(df["Parameter index"].astype(str), df["MCSE Mean"], color="#7B4FA3")
    axes[1,0].set_title("MCSE of posterior mean")
    axes[1,1].bar(df["Parameter index"].astype(str), df["MCSE SD"], color="#F28E2B")
    axes[1,1].set_title("MCSE of posterior SD")
    for ax in axes.ravel():
        ax.grid(axis="y", alpha=0.25)
        ax.tick_params(axis='x', rotation=35)
    fig.tight_layout()
    if path is not None:
        fig.savefig(path, bbox_inches="tight")
        plt.close(fig)


In [ ]:
# ============================================================
# CELL 43: SGHMC Dynamics Diagnostic Plot
# ============================================================

def summarize_chain_diagnostics(diag_df: pd.DataFrame, cfg: ExperimentConfig) -> pd.DataFrame:
    rows = []
    for chain, sub in diag_df.groupby("chain"):
        retained = sub[sub["iteration"] > cfg.sghmc_burn_in]
        rows.append({
            "Chain": chain,
            "Mean update norm": retained["update_norm"].mean(),
            "Std update norm": retained["update_norm"].std(),
            "Mean gradient norm": retained["gradient_norm"].mean(),
            "Gradient noise variance": retained["gradient_norm"].var(),
            "Momentum norm": retained["momentum_norm"].mean(),
            "Energy drift proxy": retained["loss"].diff().abs().mean(),
        })
    return pd.DataFrame(rows)


def plot_sghmc_dynamics(dynamics_df: pd.DataFrame, path: Optional[Path] = None):
    metric_cols = [c for c in dynamics_df.columns if c != "Chain"]
    plot_df = dynamics_df.melt(id_vars="Chain", value_vars=metric_cols, var_name="Metric", value_name="Value")
    plt.figure(figsize=(11, 5))
    if sns is not None:
        sns.barplot(data=plot_df, x="Metric", y="Value", hue="Chain", palette="Set2")
    plt.xticks(rotation=25, ha="right")
    plt.ylabel("Diagnostic value")
    plt.grid(axis="y", alpha=0.25)
    plt.legend(frameon=True)
    plt.tight_layout()
    if path is not None:
        plt.savefig(path, bbox_inches="tight")
        plt.close()


In [ ]:
# ============================================================
# CELL 44: Runtime and Memory Figures
# ============================================================

def plot_runtime_memory(runtime_df: pd.DataFrame, path: Optional[Path] = None):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.8))
    axes[0].bar(runtime_df["Method"], runtime_df["Inference runtime (s)"], color="#2B6CB0")
    axes[0].set_yscale("log")
    axes[0].set_ylabel("Seconds")
    axes[0].set_title("Inference runtime")
    axes[1].bar(runtime_df["Method"], runtime_df["Inference Memory (MB)"], color="#D62728")
    axes[1].set_yscale("log")
    axes[1].set_ylabel("MB")
    axes[1].set_title("Inference memory")
    for ax in axes:
        ax.tick_params(axis='x', rotation=25)
        ax.grid(axis="y", alpha=0.25)
    fig.tight_layout()
    if path is not None:
        fig.savefig(path, bbox_inches="tight")
        plt.close(fig)


In [ ]:
# ============================================================
# CELL 45: Metric-Wise Ranking Heatmap
# ============================================================

def metric_direction(metric: str) -> str:
    lower_better = {"ECE", "MCE", "NLL", "Brier", "Mean Prediction Entropy", "Std Dev Prediction Entropy", "Inference runtime (s)", "Inference Memory (MB)", "Misclassification FPR@95%TPR (OoD)"}
    return "min" if metric in lower_better else "max"


def build_rank_table(metrics_df: pd.DataFrame, extra_df: Optional[pd.DataFrame] = None) -> pd.DataFrame:
    df = metrics_df.copy()
    if extra_df is not None:
        df = df.merge(extra_df, on=["Dataset", "Method"], how="left")
    metric_cols = [c for c in df.columns if c not in ["Dataset", "Method"]]
    rows = []
    for dataset, sub in df.groupby("Dataset"):
        row_block = sub[["Method"]].copy()
        row_block.insert(0, "Dataset", dataset)
        for col in metric_cols:
            ascending = metric_direction(col) == "min"
            row_block[col] = sub[col].rank(method="average", ascending=ascending).values
        rows.append(row_block)
    return pd.concat(rows, ignore_index=True)


def plot_rank_heatmap(rank_df: pd.DataFrame, dataset_name: str, path: Optional[Path] = None):
    sub = rank_df[rank_df["Dataset"] == dataset_name].set_index("Method").drop(columns=["Dataset"])
    plt.figure(figsize=(12, 4.5))
    if sns is not None:
        sns.heatmap(sub, annot=True, fmt=".0f", cmap="YlOrRd_r", cbar_kws={"label": "Rank (1 = best)"})
    else:
        plt.imshow(sub.values, aspect="auto")
        plt.xticks(range(sub.shape[1]), sub.columns, rotation=45, ha="right")
        plt.yticks(range(sub.shape[0]), sub.index)
    plt.tight_layout()
    if path is not None:
        plt.savefig(path, bbox_inches="tight")
        plt.close()


In [ ]:
# ============================================================
# CELL 46: Save Core Tables
# ============================================================

def save_core_tables(result: Dict[str, Any], prefix: str, cfg: ExperimentConfig):
    if not cfg.save_tables:
        return
    safe_to_csv(result["metrics"], table_dir / f"{prefix}_classification_calibration_metrics.csv")
    safe_to_csv(result["ood_metrics"], table_dir / f"{prefix}_ood_misclassification_metrics.csv")
    safe_to_csv(result["train_history"], table_dir / f"{prefix}_training_history.csv")
    dynamics = summarize_chain_diagnostics(result["sghmc_diagnostics"], cfg)
    safe_to_csv(result["sghmc_diagnostics"], table_dir / f"{prefix}_sghmc_iteration_diagnostics.csv")
    safe_to_csv(dynamics, table_dir / f"{prefix}_sghmc_chain_diagnostics.csv")
    posterior_summary = summarize_sghmc_states(result["posterior_states"], max_parameters=20)
    safe_to_csv(posterior_summary, table_dir / f"{prefix}_posterior_parameter_summary.csv")
    for method, prob in result["probabilities"].items():
        rel = reliability_table(result["y_test"], prob, cfg.calibration_bins)
        safe_to_csv(rel, table_dir / f"{prefix}_{method.replace(' ', '_').replace('+','plus')}_reliability_bins.csv")


In [ ]:
# ============================================================
# CELL 47: Save Core Figures
# ============================================================

def save_core_figures(result: Dict[str, Any], prefix: str, cfg: ExperimentConfig):
    if not cfg.save_figures:
        return
    metrics_df = result["metrics"]
    dataset_name = result["graph"]["name"]
    y_test = result["y_test"]
    prob_dict = result["probabilities"]
    plot_classification_bars(metrics_df, dataset_name, fig_dir / f"{prefix}_classification_bars.png")
    plot_calibration_errors(metrics_df, fig_dir / f"{prefix}_calibration_errors.png")
    plot_entropy_metrics(metrics_df, fig_dir / f"{prefix}_entropy_metrics.png")
    plot_joint_reliability(y_test, prob_dict, fig_dir / f"{prefix}_joint_reliability.png", cfg.calibration_bins)
    plot_roc_pr_curves(y_test, prob_dict, fig_dir / f"{prefix}_roc_pr_curves.png")
    plot_confusion_matrices(y_test, prob_dict, fig_dir / f"{prefix}_confusion_matrices.png")
    posterior_summary = summarize_sghmc_states(result["posterior_states"], max_parameters=20)
    plot_sghmc_trace_grid(result["posterior_states"], [0,1,2,3,4], fig_dir / f"{prefix}_sghmc_trace_grid.png")
    plot_sghmc_density_grid(result["posterior_states"], [0,1,2,3,4], fig_dir / f"{prefix}_sghmc_density_grid.png")
    plot_posterior_hdi(posterior_summary, fig_dir / f"{prefix}_posterior_hdi.png")
    plot_sghmc_diagnostics(posterior_summary, fig_dir / f"{prefix}_sghmc_diagnostics.png")
    dynamics = summarize_chain_diagnostics(result["sghmc_diagnostics"], cfg)
    plot_sghmc_dynamics(dynamics, fig_dir / f"{prefix}_sghmc_dynamics.png")


In [ ]:
# ============================================================
# CELL 48: Experiment Orchestration Helpers
# ============================================================

def run_all_configured_datasets(cfg: ExperimentConfig, specs: List[DatasetSpec]):
    all_results = {}
    metric_frames = []
    ood_frames = []
    for spec in specs:
        result = run_sgh_graphbnn_for_dataset(spec, cfg, cfg.seed)
        prefix = spec.name.replace(" ", "_").replace("/", "_")
        save_core_tables(result, prefix, cfg)
        save_core_figures(result, prefix, cfg)
        all_results[spec.name] = result
        metric_frames.append(result["metrics"])
        ood_frames.append(result["ood_metrics"])
    combined_metrics = pd.concat(metric_frames, ignore_index=True)
    combined_ood = pd.concat(ood_frames, ignore_index=True)
    safe_to_csv(combined_metrics, table_dir / "combined_metrics.csv")
    safe_to_csv(combined_ood, table_dir / "combined_ood_metrics.csv")
    return all_results, combined_metrics, combined_ood


In [ ]:
# ============================================================
# CELL 49: Artifact Manifest
# ============================================================

def write_manifest(cfg: ExperimentConfig):
    files = []
    for root, _, filenames in os.walk(cfg.output_dir):
        for fn in filenames:
            p = Path(root) / fn
            files.append({"path": str(p), "size_bytes": p.stat().st_size})
    manifest = {
        "framework": FRAMEWORK_NAME,
        "configuration": asdict(cfg),
        "files": files,
    }
    save_json(manifest, output_dir / "artifact_manifest.json")


In [ ]:
# ============================================================
# CELL 50: Main Execution Block
# ============================================================
# Set the desired run flags to True after dataset paths are configured.

RUN_SINGLE_SEED_EXPERIMENT = False
RUN_FIVE_SEED_EXPERIMENT = False
RUN_DATASET_PROFILING = False

if RUN_DATASET_PROFILING:
    dataset_profiles = profile_all_configured_datasets(DATASET_SPECS, cfg)

if RUN_SINGLE_SEED_EXPERIMENT:
    all_results, combined_metrics, combined_ood = run_all_configured_datasets(cfg, DATASET_SPECS)
    write_manifest(cfg)

if RUN_FIVE_SEED_EXPERIMENT:
    five_seed_frames = []
    five_seed_summary_frames = []
    statistical_frames = []
    for spec in DATASET_SPECS:
        seed_metric_df, seed_summary_df = run_five_seed_experiment(spec, cfg)
        prefix = spec.name.replace(" ", "_").replace("/", "_")
        safe_to_csv(seed_metric_df, table_dir / f"{prefix}_five_seed_metric_values.csv")
        safe_to_csv(seed_summary_df, table_dir / f"{prefix}_five_seed_metric_summary.csv")
        comparison_df = save_statistical_comparisons(seed_metric_df, cfg)
        safe_to_csv(comparison_df, table_dir / f"{prefix}_sgh_graphbnn_vs_baselines_statistical_comparison.csv")
        five_seed_frames.append(seed_metric_df)
        five_seed_summary_frames.append(seed_summary_df)
        statistical_frames.append(comparison_df)

    all_five_seed_metrics = pd.concat(five_seed_frames, ignore_index=True)
    all_five_seed_summary = pd.concat(five_seed_summary_frames, ignore_index=True)
    all_statistical_comparisons = pd.concat(statistical_frames, ignore_index=True)
    safe_to_csv(all_five_seed_metrics, table_dir / "all_datasets_five_seed_metric_values.csv")
    safe_to_csv(all_five_seed_summary, table_dir / "all_datasets_five_seed_metric_summary.csv")
    safe_to_csv(all_statistical_comparisons, table_dir / "all_datasets_sgh_graphbnn_vs_baselines_statistical_comparison.csv")
    write_manifest(cfg)


---

## Notes for use

1. Place the NF-ToN-IoT-v3 and CICIoT2023 CSV files in the configured dataset directory.
2. Update `DatasetSpec` paths if the files are stored elsewhere.
3. Set `RUN_FULL_EXPERIMENT = True` in the final cell to execute the complete training, SGHMC posterior inference, and supplementary artifact generation pipeline.
4. The notebook saves tables to `outputs_sgh_graphbnn/tables/` and figures to `outputs_sgh_graphbnn/figures/`.